In [2]:
import numpy as np
import pandas as pd
import src.csv_io as csv_io 
from pathlib import Path
from src.config import RAW_DIR

Question 1: Trong số các khách hàng có nhiều hơn một đơn hàng, trung vị số ngày giữa hai lần
mua liên tiếp (inter-order gap) xấp xỉ là bao nhiêu? (Tính từ orders.csv)

In [3]:
orders = csv_io.read_csv_file(RAW_DIR / "orders.csv")
orders["order_date"] = pd.to_datetime(orders["order_date"])
orders = orders.sort_values(["customer_id", "order_date", "order_id"])

inter_order_gap_days = (
    orders.groupby("customer_id")["order_date"]
    .diff()
    .dt.days
    .dropna()
)

q1_median_gap_days = float(np.median(inter_order_gap_days.to_numpy()))
print(f"Q1 answer: median inter-order gap ~ {q1_median_gap_days:.0f} days")


Q1 answer: median inter-order gap ~ 144 days


Question 2: Phân khúc sản phẩm (segment) nào trong products.csv có tỷ suất lợi nhuận gộp
trung bình cao nhất, với công thức (price − cogs)/price?

In [4]:
products = csv_io.read_csv_file(RAW_DIR / "products.csv")
products = products.copy()

products["gross_margin_rate"] = (products["price"] - products["cogs"]) / products["price"]
segment_margin = (
    products.groupby("segment", as_index=False)["gross_margin_rate"]
    .mean()
    .sort_values("gross_margin_rate", ascending=False)
)

q2_best_segment = segment_margin.iloc[0]["segment"]
q2_best_margin = float(segment_margin.iloc[0]["gross_margin_rate"])

print(f"Q2 answer: best segment = {q2_best_segment} (avg gross margin ~ {q2_best_margin:.2%})")
segment_margin


Q2 answer: best segment = Standard (avg gross margin ~ 31.34%)


,segment,gross_margin_rate
6,Standard,0.313442
5,Premium,0.285377
1,All-weather,0.284176
0,Activewear,0.265600
4,Performance,0.263650
2,Balanced,0.258038
7,Trendy,0.240758
3,Everyday,0.236343


Question 3: Trong các bản ghi trả hàng liên kết với sản phẩm thuộc danh mục Streetwear (join
returns với products theo product_id), lý do trả hàng nào xuất hiện nhiều nhất?


In [5]:
products = csv_io.read_csv_file(RAW_DIR / "products.csv")
returns = csv_io.read_csv_file(RAW_DIR / "returns.csv")

products_cp = products.copy()
returns_cp = returns.copy()

products_cp["product_id"] = products_cp["product_id"].astype(str)
returns_cp["product_id"] = returns_cp["product_id"].astype(str)

merged = pd.merge(returns_cp, products_cp, how="left", on="product_id")

streetwear_returns = merged[merged["category"] == "Streetwear"]

q3_most_returned_reason = streetwear_returns["return_reason"].value_counts().idxmax()

print(f"Q3 answer: Lý do trả hàng phổ biến nhất của Streetwear = {q3_most_returned_reason}")

Q3 answer: Lý do trả hàng phổ biến nhất của Streetwear = wrong_size


Question 4: Trong web_traffic.csv, nguồn truy cập (traffic_source) nào có tỷ lệ thoát trung
bình (bounce_rate) thấp nhất trên tất cả các ngày xuất hiện nguồn đó trong cột traffic_source?

In [6]:
web_traffic = csv_io.read_csv_file(RAW_DIR / "web_traffic.csv")

q4_source_bounce = (
    web_traffic.groupby("traffic_source", as_index=False)["bounce_rate"]
    .mean()
    .sort_values("bounce_rate", ascending=True)
)

q4_best_source = q4_source_bounce.iloc[0]["traffic_source"]
q4_lowest_bounce = float(q4_source_bounce.iloc[0]["bounce_rate"])

print(f"Q4 answer: lowest-average bounce_rate source = {q4_best_source} (~ {q4_lowest_bounce:.6f})")
q4_source_bounce


Q4 answer: lowest-average bounce_rate source = email_campaign (~ 0.004458)


,traffic_source,bounce_rate
1,email_campaign,0.004458
5,social_media,0.004476
3,paid_search,0.004478
4,referral,0.004499
2,organic_search,0.004504
0,direct,0.004511


Question 5: Tỷ lệ phần trăm các dòng trong order_items.csv có áp dụng khuyến mãi (tức là promo_id
không null) xấp xỉ là bao nhiêu?


In [7]:
order_items = csv_io.read_csv_file(RAW_DIR / "order_items.csv").copy()

order_items["promo_id"] = order_items["promo_id"].replace("", np.nan)

promo_applied_pct = order_items["promo_id"].notna().mean() * 100
print(f"Q5 answer: percentage of rows with promo_id applied ~ {promo_applied_pct:.2f}%")

Q5 answer: percentage of rows with promo_id applied ~ 38.66%


Question 6: Trong customers.csv, xét các khách hàng có age_group khác null, nhóm tuổi nào có số
đơn hàng trung bình trên mỗi khách hàng cao nhất? (tổng số đơn / số khách hàng trong
nhóm)


In [8]:
customers = csv_io.read_csv_file(RAW_DIR / "customers.csv").copy()
orders = csv_io.read_csv_file(RAW_DIR / "orders.csv").copy()

customers["age_group"] = customers["age_group"].replace("", np.nan)

customers_byAge = customers.groupby('age_group').size().reset_index(name='total_orders')

customers_byAge

,age_group,total_orders
0,18-24,17039
1,25-34,36342
2,35-44,31920
3,45-54,23172
4,55+,13457
